In [1]:
# physical_graph_path = "scenarios/cavia/1_26_solution_v1/physical_graph.graphml"
# app_graph_path = "scenarios/cavia/1_26_solution_v1/ms/1MMM.graphml"
# pkl_path = "scenarios/cavia/1_26_solution_v1/var_coeff_values_1MMM_slss.pkl"

from adapters.cavia.cavia_scenario_loader import CaviaScenarioLoader
from adapters.cavia.find_valid_scenarios import get_scenario_paths
from edge_sim_py.component_manager import ComponentManager

scenario_name = "2_26_solution_v107"
app_name = "2SMM"
phys_path, app_path, pkl_path = get_scenario_paths(scenario_name, app_name)
print("Loading scenario:", scenario_name)
print("Loading app:", app_name + "\n")

topology, apps, users = CaviaScenarioLoader(
    physical_graph_path=phys_path,
    app_graph_path=app_path,
    pkl_path=pkl_path,
    dist="exponential",
).build_scenario()

print("Topology:", topology)
print("Application:", apps)
print("User:", users)

Found valid_scenarios_cache.json in: /home/damiano/tesi/EdgeSimPy/adapters/cavia/valid_scenarios_cache.json
Loading scenario: 2_26_solution_v107
Loading app: 2SMM

Topology: Topology_1
Application: [Application_1, Application_2]
User: [User_1, User_2]


In [2]:
from edge_sim_py.simulator import Simulator

dataset = ComponentManager.export_scenario(save_to_file=True, file_name=scenario_name)

def my_algorithm(parameters):
    return

def static_dummy_mobility(user):
    user.coordinates_trace.append(user.coordinates)

# Instantiating the simulator
simulator = Simulator(
    dump_interval=1,
    tick_unit="seconds",
    tick_duration=1,
    stopping_criterion= lambda model: model.schedule.steps == 100,
    resource_management_algorithm=my_algorithm,
    user_defined_functions=[static_dummy_mobility]
)

# Loading our dataset
simulator.initialize(input_file=f"datasets/{scenario_name}.json")

# Running the simulation
simulator.run_model()

In [3]:
import msgpack
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 1000)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 1000)

def highlight_rows(row):
    if row["Time Step"] % 2 == 0:
        return ["background-color: #222222; color: white"] * len(row)
    else:
        return ["background-color: #333333; color: white"] * len(row)

def load_simulation_logs(logs_path="logs"):
    datasets = {}
    directory = Path(logs_path)
    
    if not directory.exists():
        print(f"Errore: La cartella {logs_path} non esiste.")
        return datasets

    for file_path in directory.glob("*.msgpack"):
        with open(file_path, "rb") as f:
            data = msgpack.unpackb(f.read(), strict_map_key=False)
            datasets[file_path.stem] = pd.DataFrame(data)
            
    return datasets

datasets = load_simulation_logs()

In [4]:
print(datasets.keys())
#datasets["Application"].copy().style.apply(highlight_rows, axis=1)
#datasets["EdgeServer"].copy().style.apply(highlight_rows, axis=1)
#d = datasets["User"].copy().style.apply(highlight_rows, axis=1)
datasets["User"].head(4).style.apply(highlight_rows, axis=1)
#datasets["Service"].copy().style.apply(highlight_rows, axis=1)
#datasets["NetworkFlow"].copy().style.apply(highlight_rows, axis=1)

dict_keys(['Application', 'DataPacket', 'User', 'EdgeServer', 'Service', 'NetworkSwitch'])


,Object,Time Step,Instance ID,Coordinates,Base Station,Delays,Delay SLAs,Communication Paths,Making Requests,Access History
0,User_1,0,1,"[15, 1]","BaseStation_8 ([15, 1])",{'1': None},{'1': 75.203},{},{'1': {'1': True}},"{'1': [{'start': 1, 'end': 1, 'duration': 1, 'waiting_time': 0, 'access_time': 0, 'interval': 100, 'next_access': 102}]}"
1,User_2,0,2,"[8, 0]","BaseStation_22 ([8, 0])",{'2': None},{'2': 75.203},{},{'2': {'1': True}},"{'2': [{'start': 1, 'end': 1, 'duration': 1, 'waiting_time': 0, 'access_time': 0, 'interval': 100, 'next_access': 102}]}"
2,User_1,1,1,"[15, 1]","BaseStation_8 ([15, 1])",{'1': 45},{'1': 75.203},"{'1': [[8], [8], [8], [8, 9, 16], [16, 9, 15, 12], [12, 6, 4, 8]]}","{'1': {'1': True, '2': False}}","{'1': [{'start': 1, 'end': 1, 'duration': 1, 'waiting_time': 0, 'access_time': 1, 'interval': 100, 'next_access': 102}]}"
3,User_2,1,2,"[8, 0]","BaseStation_22 ([8, 0])",{'2': 41},{'2': 75.203},"{'2': [[22], [22], [22, 0, 16], [16, 9, 15, 12], [12, 6, 4, 8]]}","{'2': {'1': True, '2': False}}","{'2': [{'start': 1, 'end': 1, 'duration': 1, 'waiting_time': 0, 'access_time': 1, 'interval': 100, 'next_access': 102}]}"


In [5]:
from edge_sim_py.components.data_packet import DataPacket
print(len(DataPacket.all()))

2


In [6]:
#datasets["DataPacket"].copy().style.apply(highlight_rows, axis=1)
df_finished = datasets["DataPacket"][datasets["DataPacket"]["Status"] == "finished"].copy()
#df_finished.head(1).style.apply(highlight_rows, axis=1)
df_finished.columns


Index(['Object', 'Time Step', 'Id', 'User', 'Application', 'Size', 'Status', 'Queue Delay', 'Transmission Delay', 'Processing Delay', 'Propagation Delay', 'Total Delay', 'Total Path', 'Hops'], dtype='str')

In [7]:
#df_finished[["Time Step", "Total Delay"]].head(1).style.apply(highlight_rows, axis=1)
df_finished.drop(columns=["Hops"]).head(2).style.apply(highlight_rows, axis=1)

,Object,Time Step,Id,User,Application,Size,Status,Queue Delay,Transmission Delay,Processing Delay,Propagation Delay,Total Delay,Total Path
121,DataPacket_2,61,2,2,2,1,finished,0,8,49,41,98,"[[22], [22], [22, 0, 16], [16, 9, 15, 12], [12, 6, 4, 8]]"
122,DataPacket_1,62,1,1,1,1,finished,0,8,49,45,102,"[[8], [8], [8], [8, 9, 16], [16, 9, 15, 12], [12, 6, 4, 8]]"


In [8]:
df_finished[["Time Step", "Application", "Hops"]].head(2).style.apply(highlight_rows, axis=1)

,Time Step,Application,Hops
121,61,2,"[{'hop_index': 0, 'link_index': 0, 'source': 22, 'target': 22, 'start_time': 1, 'end_time': 1, 'queue_delay': 0, 'transmission_delay': 0, 'processing_delay': 0, 'propagation_delay': 0, 'min_bandwidth': 0, 'max_bandwidth': 0, 'avg_bandwidth': 0, 'data_input': 1, 'data_output': 1}, {'hop_index': 1, 'link_index': 0, 'source': 22, 'target': 22, 'start_time': 1, 'end_time': 6, 'queue_delay': 0, 'transmission_delay': 0, 'processing_delay': 5, 'propagation_delay': 0, 'min_bandwidth': 0, 'max_bandwidth': 0, 'avg_bandwidth': 0, 'data_input': 1, 'data_output': 1}, {'hop_index': 2, 'link_index': 0, 'source': 22, 'target': 0, 'start_time': 8, 'end_time': 9, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 0, 'propagation_delay': 8, 'min_bandwidth': 50.0, 'max_bandwidth': 50.0, 'avg_bandwidth': 50.0, 'data_input': 1, 'data_output': 1}, {'hop_index': 2, 'link_index': 1, 'source': 0, 'target': 16, 'start_time': 9, 'end_time': 33, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 23, 'propagation_delay': 4, 'min_bandwidth': 64.0, 'max_bandwidth': 64.0, 'avg_bandwidth': 64.0, 'data_input': 1, 'data_output': 1}, {'hop_index': 3, 'link_index': 0, 'source': 16, 'target': 9, 'start_time': 33, 'end_time': 34, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 0, 'propagation_delay': 12, 'min_bandwidth': 88.0, 'max_bandwidth': 88.0, 'avg_bandwidth': 88.0, 'data_input': 1, 'data_output': 1}, {'hop_index': 3, 'link_index': 1, 'source': 9, 'target': 15, 'start_time': 34, 'end_time': 35, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 0, 'propagation_delay': 3, 'min_bandwidth': 58.0, 'max_bandwidth': 58.0, 'avg_bandwidth': 58.0, 'data_input': 1, 'data_output': 1}, {'hop_index': 3, 'link_index': 2, 'source': 15, 'target': 12, 'start_time': 35, 'end_time': 57, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 21, 'propagation_delay': 5, 'min_bandwidth': 34.0, 'max_bandwidth': 34.0, 'avg_bandwidth': 34.0, 'data_input': 1, 'data_output': 1}, {'hop_index': 4, 'link_index': 0, 'source': 12, 'target': 6, 'start_time': 57, 'end_time': 58, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 0, 'propagation_delay': 3, 'min_bandwidth': 42.0, 'max_bandwidth': 42.0, 'avg_bandwidth': 42.0, 'data_input': 1, 'data_output': 1}, {'hop_index': 4, 'link_index': 1, 'source': 6, 'target': 4, 'start_time': 58, 'end_time': 59, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 0, 'propagation_delay': 3, 'min_bandwidth': 61.0, 'max_bandwidth': 61.0, 'avg_bandwidth': 61.0, 'data_input': 1, 'data_output': 1}, {'hop_index': 4, 'link_index': 2, 'source': 4, 'target': 8, 'start_time': 59, 'end_time': 60, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 0, 'propagation_delay': 3, 'min_bandwidth': 58.0, 'max_bandwidth': 58.0, 'avg_bandwidth': 58.0, 'data_input': 1, 'data_output': 1}]"
122,62,1,"[{'hop_index': 0, 'link_index': 0, 'source': 8, 'target': 8, 'start_time': 1, 'end_time': 1, 'queue_delay': 0, 'transmission_delay': 0, 'processing_delay': 0, 'propagation_delay': 0, 'min_bandwidth': 0, 'max_bandwidth': 0, 'avg_bandwidth': 0, 'data_input': 1, 'data_output': 1}, {'hop_index': 1, 'link_index': 0, 'source': 8, 'target': 8, 'start_time': 1, 'end_time': 6, 'queue_delay': 0, 'transmission_delay': 0, 'processing_delay': 5, 'propagation_delay': 0, 'min_bandwidth': 0, 'max_bandwidth': 0, 'avg_bandwidth': 0, 'data_input': 1, 'data_output': 1}, {'hop_index': 2, 'link_index': 0, 'source': 8, 'target': 8, 'start_time': 6, 'end_time': 6, 'queue_delay': 0, 'transmission_delay': 0, 'processing_delay': 0, 'propagation_delay': 0, 'min_bandwidth': 0, 'max_bandwidth': 0, 'avg_bandwidth': 0, 'data_input': 1, 'data_output': 1}, {'hop_index': 3, 'link_index': 0, 'source': 8, 'target': 9, 'start_time': 9, 'end_time': 10, 'queue_delay': 0, 'transmission_delay': 1, 'processing_delay': 0, 'propagation_delay': 4, 'min_bandwidth': 60.0, 'max_bandwidth': 60.0, 'avg_ba

In [9]:
# Estraiamo gli hops del primo pacchetto finito
hops_list = df_finished.iloc[0]["Hops"]
df_hops = pd.DataFrame(hops_list)

# Selezioniamo le colonne chiave per capire il percorso e i ritardi
cols = ["hop_index", "link_index", "source", "target", "propagation_delay", "processing_delay", "transmission_delay", "data_input", "data_output"]
df_hops[cols]

,hop_index,link_index,source,target,propagation_delay,processing_delay,transmission_delay,data_input,data_output
0,0,0,22,22,0,0,0,1,1
1,1,0,22,22,0,5,0,1,1
2,2,0,22,0,8,0,1,1,1
3,2,1,0,16,4,23,1,1,1
4,3,0,16,9,12,0,1,1,1
5,3,1,9,15,3,0,1,1,1
6,3,2,15,12,5,21,1,1,1
7,4,0,12,6,3,0,1,1,1
8,4,1,6,4,3,0,1,1,1
9,4,2,4,8,3,0,1,1,1


In [10]:
hops_list1 = df_finished.iloc[1]["Hops"]
df_hops1 = pd.DataFrame(hops_list1)
df_hops1[cols]

,hop_index,link_index,source,target,propagation_delay,processing_delay,transmission_delay,data_input,data_output
0,0,0,8,8,0,0,0,1,1
1,1,0,8,8,0,5,0,1,1
2,2,0,8,8,0,0,0,1,1
3,3,0,8,9,4,0,1,1,1
4,3,1,9,16,12,23,1,1,1
5,4,0,16,9,12,0,1,1,1
6,4,1,9,15,3,0,1,1,1
7,4,2,15,12,5,21,1,1,1
8,5,0,12,6,3,0,1,1,1
9,5,1,6,4,3,0,1,1,1
